# Anime Recommendation System using Cosine Similarity

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv('anime.csv')
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna(df['type'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mean())

In [ ]:
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

In [ ]:
df.isnull().sum()

In [ ]:
df.drop_duplicates(subset='name', keep='first', inplace=True)
df.reset_index(drop=True, inplace=True)
df.shape

In [ ]:
df.describe()

In [ ]:
tfidf = TfidfVectorizer(token_pattern=r'[A-Za-z0-9\-]+')
genre_matrix = tfidf.fit_transform(df['genre'])
genre_matrix.shape

In [ ]:
type_dummies = pd.get_dummies(df['type'])
type_dummies.shape

In [ ]:
scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])
numeric_features.shape

In [ ]:
from scipy.sparse import hstack, csr_matrix

feature_matrix = hstack([
    genre_matrix,
    csr_matrix(type_dummies.values),
    csr_matrix(numeric_features)
])
feature_matrix.shape

In [ ]:
cosine_sim = cosine_similarity(feature_matrix, feature_matrix)
cosine_sim.shape

In [ ]:
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [ ]:
def recommend_anime(title, top_n=10, threshold=0.0):
    if title not in indices:
        return f"'{title}' not found in dataset"

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = [s for s in sim_scores if s[0] != idx]
    sim_scores = [s for s in sim_scores if s[1] >= threshold]
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[:top_n]

    anime_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]

    result = df.iloc[anime_indices][['name', 'genre', 'type', 'rating']].copy()
    result['similarity_score'] = scores
    return result.reset_index(drop=True)

In [ ]:
recommend_anime('Naruto', top_n=10)

In [ ]:
recommend_anime('Death Note', top_n=10, threshold=0.3)

In [ ]:
for t in [0.0, 0.2, 0.4, 0.6, 0.8]:
    res = recommend_anime('Naruto', top_n=50, threshold=t)
    print(f"Threshold: {t} -> Number of recommendations: {len(res)}")

In [ ]:
sample_titles = ['Naruto', 'Death Note', 'One Piece', 'Bleach', 'Fullmetal Alchemist: Brotherhood']

for title in sample_titles:
    if title in indices.values or title in indices:
        res = recommend_anime(title, top_n=5)
        print(f"\nTop 5 recommendations for '{title}':")
        print(res[['name', 'similarity_score']])

In [ ]:
avg_sim_scores = []
for title in sample_titles:
    if title in indices:
        res = recommend_anime(title, top_n=10)
        avg_sim_scores.append(res['similarity_score'].mean())

print("Average similarity score across sample queries:", np.mean(avg_sim_scores))

## 6. Interview Questions

**1. Can you explain the difference between user-based and item-based collaborative filtering?**

User-based collaborative filtering finds users who have similar preferences or rating patterns to the target user, and recommends items that those similar users liked but the target user has not yet interacted with. It relies on computing similarity between users based on their historical ratings or behavior.

Item-based collaborative filtering instead finds similarity between items based on how users have rated or interacted with them. Recommendations are made by identifying items that are similar to the ones a user has already liked or rated highly. Item-based methods tend to be more stable over time since item relationships change less frequently than user preferences, and they scale better when the number of users is much larger than the number of items.

**2. What is collaborative filtering, and how does it work?**

Collaborative filtering is a recommendation technique that predicts a user's interests by collecting preferences from many users and finding patterns of similarity between them. It works on the assumption that users who agreed in the past will agree in the future, or that items liked together by many users are related.

It works by building a user-item interaction matrix (such as ratings), then computing similarity either between users or between items using metrics like cosine similarity or Pearson correlation. Based on these similarities, the system predicts ratings for items a user has not yet seen and recommends the highest predicted items. Collaborative filtering does not require content information about the items themselves, but it can suffer from the cold-start problem when there is insufficient interaction data for new users or items.